# NB05 — Linear and Logistic Baseline Models

**Role:** Core  
**Manual section:** 2.3.1.a — Logistic regression baseline  
**Prerequisites:** NB04

---

## Purpose of this notebook

This notebook establishes the first predictive baseline using logistic regression. You will see how a model is trained on the modelling table, learn to read a confusion matrix in business terms, and understand precision, recall, F1 and AUC as financial decision metrics.

**Dataset:** `dataset_C_bank_attrition_core.csv` + `dataset_C_attrition_model_ready.csv`

> **Core route notebook**  
> This is the baseline notebook of the running attrition (also called churn) project.

> **Shared project file**  
> It loads the exact canonical table created in **NB04** and validated in **NB04b**:  
> `dataset_C_attrition_model_ready.csv`

> **Optional short extensions (10–15 min each)**  
> - **NB05c** for Naïve Bayes, SVM and k-NN on the same type of problem  
> - **NB05d** for a very small PyTorch neural-network example

---

## Section 1 — Regression vs Classification in Plain Language

Machine learning splits into two main families of supervised problems:

| | Regression | Classification |
|---|-----------|---------------|
| **Target** | Continuous number | Category (often binary) |
| **Example** | Predict a customer's annual spend (EUR) | Predict whether a customer will leave (yes / no) |
| **Output** | A number (e.g., 12 500.00) | A class label and/or a probability |
| **Metric** | MAE, RMSE, R² | Accuracy, precision, recall, F1, AUC |

### Our case

Our bank attrition dataset has a **binary target**: `attrition_flag` (0 = stayed, 1 = attrited).  
This is a **classification** problem.

We will spend most of our time on logistic regression for classification, but first a very
short regression demo to make the distinction concrete.

---

## Section 2 — Quick Regression Micro-Demo

To illustrate what regression means, let's use a **tiny** example: predicting a customer's
balance from their age, salary and number of products.

This is only a short demonstration — the main classification model follows in Section 3.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# Resolve data directory — works from the project root or notebooks/
_candidates = [Path("data/processed"), Path("../data/processed")]
DATA_DIR = next((p for p in _candidates if p.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Cannot locate data/processed/. "
        "Launch the notebook from the project root or the notebooks/ folder."
    )

# Load dataset
df = pd.read_csv(DATA_DIR / "dataset_C_bank_attrition_core.csv")

# Quick regression: predict balance from age, salary, num_of_products
reg_features = ["age", "estimated_salary", "num_of_products"]
mask = df["balance"] > 0  # only customers with a positive balance
X_reg = df.loc[mask, reg_features]
y_reg = df.loc[mask, "balance"]

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=42
)

reg_model = LinearRegression()
reg_model.fit(X_reg_train, y_reg_train)

y_reg_pred = reg_model.predict(X_reg_test)

print("=== Regression micro-demo: predicting customer balance ===")
print(f"MAE  (Mean Absolute Error): {mean_absolute_error(y_reg_test, y_reg_pred):,.0f} EUR")
print(f"R²   (Coefficient of determination): {r2_score(y_reg_test, y_reg_pred):.3f}")

In [ ]:
# Scatter: predicted vs actual
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_reg_test, y_reg_pred, alpha=0.15, s=10, color="steelblue")
lims = [0, y_reg_test.max()]
ax.plot(lims, lims, "--", color="red", linewidth=1, label="Perfect prediction")
ax.set_xlabel("Actual Balance (EUR)")
ax.set_ylabel("Predicted Balance (EUR)")
ax.set_title("Regression Micro-Demo: Predicted vs Actual Balance")
ax.legend()
plt.tight_layout()
plt.show()

> The R² is very low — these three variables alone do not explain balance well.  
> That is fine: the point of this demo is to show **what a regression problem looks like**
> (continuous target, numeric prediction), not to build a good balance model.

Now let's move to our **main task**: classifying attrition.

---

## Section 3 — Logistic Regression Baseline

### Recap from NB04 and NB04b

We already have a **shared canonical project table**:

- one row = one customer,
- a binary target (`attrition_flag`),
- numeric and encoded predictors,
- and a stable file reused across the attrition project.

This is the point of good project design in finance: later model notebooks should not silently redefine the dataset.

In this notebook we load:

`dataset_C_attrition_model_ready.csv`


In [ ]:
# --- Load the shared model-ready table created in NB04 and validated in NB04b ---
from sklearn.preprocessing import StandardScaler

MODEL_PATH = DATA_DIR / 'dataset_C_attrition_model_ready.csv'
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        'Shared project dataset not found. Run NB04 and NB04b first so that '
        '`dataset_C_attrition_model_ready.csv` exists.'
    )

model_ready = pd.read_csv(MODEL_PATH)

target_col = 'attrition_flag'
id_col = 'customer_id'

y = model_ready[target_col]
X = model_ready.drop(columns=[target_col, id_col])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_sc  = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns,  index=X_test.index)

print(f'Loaded canonical project table: {MODEL_PATH.name}')
print(f'Training: {X_train.shape[0]:,} rows × {X_train.shape[1]} features')
print(f'Test:     {X_test.shape[0]:,} rows × {X_test.shape[1]} features')
print(f'Churn rate — train: {y_train.mean():.3f}, test: {y_test.mean():.3f}')

### What is logistic regression?

Logistic regression is a **linear model for classification**. Despite its name, it is used
for classification, not regression.

It works by:
1. Computing a weighted sum of the features (like linear regression).
2. Passing that sum through a **sigmoid function** that converts it into a probability
   between 0 and 1.
3. Applying a **threshold** (usually 0.5) to decide the class label.

The output is both:
- A **probability** (e.g., 0.73 → 73 % chance of leaving).
- A **class label** (0 or 1, based on the threshold).

In [ ]:
from sklearn.linear_model import LogisticRegression

# Fit logistic regression on scaled features
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train_sc, y_train)

print("Logistic regression fitted successfully.")
print(f"Number of features: {X_train_sc.shape[1]}")

In [ ]:
# Predictions on the test set
y_pred = logreg.predict(X_test_sc)           # class labels (0 or 1)
y_prob = logreg.predict_proba(X_test_sc)[:, 1]  # probability of class 1 (churn)

print("First 10 predictions:")
print(f"  Class labels:   {list(y_pred[:10])}")
print(f"  Probabilities:  {[round(p, 3) for p in y_prob[:10]]}")
print(f"  Actual values:  {list(y_test.values[:10])}")

---

## Section 4 — Evaluation Metrics

### The confusion matrix

The confusion matrix is the foundation of all classification metrics. It counts four types
of outcomes:

|  | Predicted Stayed (0) | Predicted Attrited (1) |
|--|---------------------|----------------------|
| **Actually Stayed (0)** | True Negative (TN) | False Positive (FP) |
| **Actually Attrited (1)** | False Negative (FN) | True Positive (TP) |

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(f"  TN={tn:4d}   FP={fp:4d}")
print(f"  FN={fn:4d}   TP={tp:4d}")

In [ ]:
# Key metrics
metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1 Score": f1_score(y_test, y_pred),
    "AUC-ROC": roc_auc_score(y_test, y_prob),
}

print("=== Baseline Logistic Regression — Test Metrics ===")
print()
for name, value in metrics.items():
    print(f"  {name:<12s} {value:.3f}")

### What do these metrics mean?

| Metric | Meaning | Business interpretation |
|--------|---------|----------------------|
| **Accuracy** | Fraction of correct predictions overall | Easy to understand, but misleading with imbalanced classes |
| **Precision** | Of those flagged for attrition, how many actually left? | How much we "waste" on false alarms |
| **Recall** | Of actual attrition cases, how many did we catch? | How many real attrition cases we miss |
| **F1 Score** | Harmonic mean of precision and recall | Balances false positives and false negatives |
| **AUC-ROC** | Ability to rank attrition cases above retained customers | Overall discrimination ability |

### Which metric matters most?

In an **attrition prevention** scenario, **recall** is often the most important metric:  
a missed attrition case (false negative) means a lost customer with no intervention attempt.

A false positive (flagging a loyal customer) is less costly — it just triggers an
unnecessary retention offer.

> **From confusion matrix to financial cost**  
> In finance, a confusion matrix is not just a technical summary — it is a cost map.  
> **False positives** may mean unnecessary retention calls or review costs.  
> **False negatives** may mean missed attrition cases, undetected fraud, or lost revenue.  
>  
> **Precision** tells us how much of our flagged population is truly relevant: it behaves like an **efficiency / risk-control** measure.  
> **Recall** tells us how much of the real positive class we actually capture: it behaves like a **business capture** measure.  
> **F1 score** gives a single compromise when we need both to be reasonable at the same time.

In [ ]:
# Full classification report
print(classification_report(y_test, y_pred, target_names=["Stayed", "Churned"]))

---

## Section 5 — Coefficients and Simple Interpretation

One of the strengths of logistic regression is that its coefficients are **directly
interpretable** (with care).

- A **positive** coefficient means that as the feature increases, the probability of
  attrition increases.
- A **negative** coefficient means the opposite.

> **Caution:** the magnitude of coefficients depends on the scale of each feature.
> A large coefficient for a feature measured in thousands of EUR is not directly comparable
> to a small coefficient for a binary flag.

In [ ]:
# Coefficients table
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": logreg.coef_[0],
}).sort_values("coefficient", ascending=False)

coef_df["direction"] = coef_df["coefficient"].apply(
    lambda c: "↑ increases churn risk" if c > 0 else "↓ decreases churn risk"
)

coef_df.reset_index(drop=True)

In [ ]:
# Visualise coefficients
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#d9534f" if c > 0 else "#5bc0de" for c in coef_df["coefficient"]]
ax.barh(coef_df["feature"], coef_df["coefficient"], color=colors, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Coefficient value")
ax.set_title("Logistic Regression — Feature Coefficients")
plt.tight_layout()
plt.show()

**Reading the chart:**

- Features pointing **right (positive)** are associated with **higher attrition risk**.
- Features pointing **left (negative)** are associated with **lower attrition risk**.
- The model has learned these associations from the training data; they are
  **correlational**, not necessarily causal.

> **Reflection:** do the directions make business sense? For example, do more complaints
> relate to higher attrition risk? Does being an active member relate to lower attrition risk?

---

## Section 6 — Train vs Test Comparison

A model that performs much better on training data than on test data is **overfitting** —
it memorises training examples instead of learning general patterns.

In [ ]:
# Train predictions
y_train_pred = logreg.predict(X_train_sc)
y_train_prob = logreg.predict_proba(X_train_sc)[:, 1]

comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1", "AUC-ROC"],
    "Train": [
        accuracy_score(y_train, y_train_pred),
        precision_score(y_train, y_train_pred),
        recall_score(y_train, y_train_pred),
        f1_score(y_train, y_train_pred),
        roc_auc_score(y_train, y_train_prob),
    ],
    "Test": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        roc_auc_score(y_test, y_prob),
    ],
})
comparison["Gap"] = comparison["Train"] - comparison["Test"]
comparison = comparison.round(3)
comparison

### Interpretation

- If the gap between train and test is **small**, the model generalises well.
- If the gap is **large**, the model may be overfitting.
- Logistic regression is a low-capacity model — it rarely overfits severely, but it may
  **underfit** (not capture complex patterns).

> **Is this baseline acceptable?** It provides a meaningful starting point. The test AUC
> tells us how well the model ranks customers who left above those who stayed overall.

---

## Section 7 — Why Baselines Are Still Useful in AI Projects

Even in projects that ultimately deploy a complex model, the baseline serves as:

1. **A fast benchmark** — built in minutes, not days.
2. **An interpretable reference** — easy to explain to stakeholders.
3. **A sanity check** — if a complex model barely beats the baseline, something may be
   wrong with the data or the problem framing.
4. **A communication tool** — "Our advanced model improves recall from X to Y compared to
   the simple baseline."

> In regulated industries like banking, a simple interpretable model may even be
> **preferred** over a black-box alternative, depending on the use case.

### Where to go next

- Open **NB05b** to compare this logistic-regression baseline against decision trees, random forest and XGBoost on the **same shared project table**.
- Open **NB05c** if you want to see short examples of Naïve Bayes, SVM and k-NN.
- Open **NB05d** if you want a compact PyTorch illustration of supervised neural networks.


---

## Section 8 — Transition to Non-Linear Models

### Where does logistic regression fall short?

Logistic regression assumes a **linear relationship** between features and the log-odds of
the outcome. This means:
- It cannot capture complex **interactions** between features (e.g., high age + low balance
  combined).
- It cannot model **non-linear patterns** (e.g., attrition risk that peaks at a specific age
  range, not at extremes).

### What comes next?

In the following notebooks we will explore models that can handle non-linearity:

| Model family | Key idea |
|-------------|----------|
| **Decision trees** | Split the data with yes/no rules |
| **Random forests** | Average many trees to reduce error |
| **XGBoost** | Build trees sequentially, each correcting the previous |

The question we will answer is:

> *"Does a more complex model add real, measurable value compared to our logistic
> regression baseline?"*

That question only makes sense because we now have a trustworthy baseline to compare against.

---

### Self-practice tasks

1. **Change the threshold:** instead of 0.5, try `(y_prob >= 0.3).astype(int)` and observe
   how precision and recall change. Why?
2. **Metric argument:** write 2–3 sentences explaining which metric matters most for this
   attrition case and why.
3. **Feature removal:** drop one predictor (e.g., `estimated_salary`), re-fit the model,
   and check if performance changes materially.
4. **Plain-English summary:** explain in simple language why a baseline is useful before
   applying XGBoost.

---

*End of Notebook 05 — Linear and Logistic Baseline Models*

---

**Project chain:** NB04 (build table) → NB04b (treat data) → **NB05 (baseline)** → NB05b (trees) → NB06 (validate & interpret)  
**Current position:** NB05